<h2> Imports & Configuration </h2>

> 🧭 **Modern Data Engineering Ecosystem Navigation**
> - 📘 **Architecture & Concept:** [Data System Design — Storage Optimization & Clustering](https://github.com/santoshjammi29/data-system-design-handbook/tree/main/modules/04_storage_optimization_and_clustering)
> - 💻 **Hands-On Recipe:** [PySpark Handbook — Notebook 08_03: Data Skew Mitigation](https://github.com/santoshjammi29/pyspark-handbook/blob/main/notebooks/08_performance_and_tuning/08_03_data_skew_mitigation_salting.py)
> - 🔬 **Empirical Lab:** [Spark Experiments — 1_data_skew: AQE & Broadcast Join](https://github.com/santoshjammi29/spark-experiments/blob/main/spark/1_data_skew/3_solving_data_skew_aqe_broadcast.ipynb) *(You are here)*
> - 🚨 **Incident Runbook:** [PySpark Runbook 03: Data Skew & Two-Phase Salting](https://github.com/santoshjammi29/pyspark-handbook/blob/main/runbooks/03_data_skew_and_two_phase_salting.md)
> - 🎯 **Interview Defense:** [Interview Handbook — Pillar 01: Spark & Databricks Q12](https://github.com/santoshjammi29/data-engineering-interview-handbook/blob/main/pillars/01_spark_and_databricks.md#q12)
> - 🌐 **Interactive Web Practice:** [Practice on moderndataengineer.uk](https://moderndataengineer.uk/practice/pyspark?topic=data-skew)
> - 📐 **Architecture Blueprint:** [Data Skew Mitigation & Key Salting Architecture](https://moderndataengineer.uk/whiteboards?id=data_skew_salting_repartition)


In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

import warnings  # noqa: E402
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
import time  # noqa: E402

In [ ]:
import pyspark.sql.functions as F  # noqa: E402
from pyspark.sql import SparkSession  # noqa: E402

In [ ]:
spark = SparkSession.builder.master("local[4]").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

In [ ]:
# spark.conf.set("spark.sql.shuffle.partitions", "3")
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Join Skews

In [ ]:
transactions_file = "../../data/data_skew/transactions.parquet"
customer_file = "../../data/data_skew/customers.parquet"

df_transactions = spark.read.parquet(transactions_file)
df_customers = spark.read.parquet(customer_file)

In [ ]:
df_transactions.printSchema()
df_transactions.show(5, False)

In [ ]:
df_customers.printSchema()
df_customers.show(5, False)

In [ ]:
(
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .show(5, False)
)

### 📊 Visual Evidence: Measured Key Skew Distribution
![Measured Key Distribution: Severe Skew Outlier](../../docs/images/generated/skew_key_distribution.png)

> **Key Observation**:
> - Customer `C0YDPQWPBJ` accounts for **17,539,732 rows (44.08%)** of the 39,790,092 total transaction records.
> - The remaining 4,999 customers average only ~4,451 rows each.
> - In standard hash partitioning (`hash(cust_id) % num_partitions`), all 17.5M records land on a single partition, creating an extreme straggler task.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../../docs/images/generated/skew_key_distribution.png"))


In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [ ]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

# Using AQE

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewedJoin.enabled", "true")

In [ ]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [ ]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

# Using Broadcast Joins

In [ ]:
# 10MB = 10485760 Bytes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

In [ ]:
df_txn_details = (
    df_transactions.join(
        F.broadcast(df_customers),
        on="cust_id",
        how="inner"
    )
)

In [ ]:
start_time = time.time()
df_txn_details.count()
print(f"time taken: {time.time() - start_time}")

### 📈 Measured Benchmark Comparison: Skew Mitigation Strategies
![Measured Join Execution Runtime Comparison](../../docs/images/generated/skew_mitigation_benchmark.png)

| Join Strategy | Spark Configuration | Measured Runtime | Speedup vs SMJ | Bottleneck / Trade-off |
| :--- | :--- | :--- | :--- | :--- |
| **SortMergeJoin (Baseline)** | `spark.sql.adaptive.enabled = false` | **11.32s** | Baseline (1.0x) | Single executor core blocked processing 17.5M rows; high disk spill risk |
| **SortMergeJoin + AQE** | `spark.sql.adaptive.enabled = true`<br>`skewJoin.enabled = true` | **9.62s** | **15.0% faster** | Automatically splits skewed partition into smaller sub-partitions |
| **BroadcastHashJoin (BHJ)** | `spark.sql.autoBroadcastJoinThreshold = 10MB` | **2.78s** | **75.5% faster (4.07x)** | Zero shuffle required; dimension table replicated to each executor |

#### 💡 Diagnostic Takeaways:
1. **SortMergeJoin Stragglers**: When shuffle partitions contain severe key skew, the job runtime is bounded by the slowest task processing the hot key.
2. **AQE Skew Join**: Spark 3.x Adaptive Query Execution detects partition sizes post-shuffle and splits oversized partitions into sub-partitions, balancing task workload dynamically without changing user code.
3. **Broadcast Hash Join Dominance**: When the dimension table (`customers`) is small (5,000 rows, ~120 KB), broadcasting it entirely eliminates the shuffle phase, cutting runtime from 11.32s down to 2.78s.


In [ ]:
from IPython.display import Image, display
display(Image(filename="../../docs/images/generated/skew_mitigation_benchmark.png"))


In [ ]:
# spark.stop()